In [1]:
!pip install tokenizers tqdm

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
import bz2
import xml.etree.ElementTree as ET
import urllib.request
import os
import math
from tqdm import tqdm

In [3]:
urls = [
    "https://dumps.wikimedia.org/other/mediawiki_content_current/enwiktionary/2026-04-01/xml/bzip2/enwiktionary-2026-04-01-p7p11575490.xml.bz2",
    "https://dumps.wikimedia.org/other/mediawiki_content_current/enwikiquote/2026-04-01/xml/bzip2/enwikiquote-2026-04-01-p1p305881.xml.bz2",
    "https://dumps.wikimedia.org/other/mediawiki_content_current/enwikibooks/2026-04-01/xml/bzip2/enwikibooks-2026-04-01-p1p482395.xml.bz2",
    "https://dumps.wikimedia.org/other/mediawiki_content_current/simplewiki/2026-04-01/xml/bzip2/simplewiki-2026-04-01-p1p1261009.xml.bz2",                    
]

os.makedirs("data", exist_ok=True)
txt_corpus_path = "data/corpus.txt"

def download_and_extract_text(urls, output_file):
    """Downloads bz2 wiki dumps and extracts basic text to a single file."""
    print("Downloading and extracting Wikipedia text...")
    with open(output_file, "w", encoding="utf-8") as out_f:
        for url in urls:
            filename = os.path.join("data", url.split('/')[-1])
            if not os.path.exists(filename):
                print(f"Downloading {filename}...")
                urllib.request.urlretrieve(url, filename)
            
            print(f"Extracting {filename}...")
            # Stream the bz2 file to save RAM
            with bz2.open(filename, "rt", encoding="utf-8") as bz2_file:
                try:
                    context = ET.iterparse(bz2_file, events=("end",))
                    for event, elem in context:
                        # Find tags ending with 'text' (ignoring namespaces)
                        if elem.tag.endswith('text') and elem.text:
                            # Basic cleanup (in a real scenario, use mwparserfromhell)
                            text = elem.text.strip()
                            if len(text) > 50: # Skip tiny entries
                                out_f.write(text + "\n\n")
                        elem.clear() # Free memory
                except ET.ParseError:
                    print(f"Reached end of valid XML in {filename}")

download_and_extract_text(urls, txt_corpus_path)

Extracting data/enwiktionary-2026-04-01-p7p11575490.xml.bz2...
Extracting data/enwikiquote-2026-04-01-p1p305881.xml.bz2...
Extracting data/enwikibooks-2026-04-01-p1p482395.xml.bz2...
Extracting data/simplewiki-2026-04-01-p1p1261009.xml.bz2...


In [4]:
# tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
# tokenizer.pre_tokenizer = Whitespace()

# Train the tokenizer on our extracted corpus
# trainer = BpeTrainer(special_tokens=["[UNK]", "[PAD]", "[BOS]", "[EOS]"], vocab_size=32000)
# print("Training Tokenizer...")
# tokenizer.train(files=[txt_corpus_path], trainer=trainer)

# Save the tokenizer in a modern format
# tokenizer.save("tokenizer.json")
# print("Tokenizer saved!")

In [5]:
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = Whitespace()
trainer = BpeTrainer(special_tokens=["[UNK]", "[PAD]", "[BOS]", "[EOS]"], vocab_size=32000)

# 1. Create a generator to yield text in chunks (saves RAM)
def batch_iterator(batch_size=10000):
    with open(txt_corpus_path, "r", encoding="utf-8") as f:
        batch = []
        for line in f:
            line = line.strip()
            if line:
                batch.append(line)
            if len(batch) == batch_size:
                yield batch
                batch = []
        if batch:
            yield batch

print("Training Tokenizer via streaming...")
# 2. Train using the iterator instead of the file path
tokenizer.train_from_iterator(batch_iterator(), trainer=trainer)

tokenizer.save("tokenizer.json")
print("Tokenizer saved!")

Training Tokenizer via streaming...



Tokenizer saved!


In [6]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout_prob):
        super().__init__()
        self.c_attn = nn.Linear(d_model, 3 * d_model, bias=False)
        self.c_proj = nn.Linear(d_model, d_model, bias=False)
        self.n_heads = n_heads
        self.d_model = d_model
        self.dropout = nn.Dropout(dropout_prob)

    def forward(self, x):
        B, T, C = x.size()
        # Calculate query, key, values
        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.d_model, dim=2)
        k = k.view(B, T, self.n_heads, C // self.n_heads).transpose(1, 2)
        q = q.view(B, T, self.n_heads, C // self.n_heads).transpose(1, 2)
        v = v.view(B, T, self.n_heads, C // self.n_heads).transpose(1, 2)

        # Causal Mask (prevents looking at future tokens)
        mask = torch.tril(torch.ones(T, T)).view(1, 1, T, T).to(x.device)
        
        # Scaled Dot-Product Attention
        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
        att = att.masked_fill(mask == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        att = self.dropout(att)
        
        y = att @ v # (B, nh, T, hs)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.c_proj(y)

class GPTBlock(nn.Module):
    def __init__(self, d_model, n_heads, dropout_prob):
        super().__init__()
        self.ln_1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, dropout_prob)
        self.ln_2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model),
            nn.Dropout(dropout_prob),
        )

    def forward(self, x):
        x = x + self.attn(self.ln_1(x)) # Residual Add + Pre-Norm
        x = x + self.mlp(self.ln_2(x))  # Residual Add + Pre-Norm
        return x

class GPT10M(nn.Module):
    def __init__(self, vocab_size=32000, d_model=256, n_heads=8, n_layers=2, max_seq_len=512, dropout_prob=0.1):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_seq_len, d_model)
        self.drop = nn.Dropout(dropout_prob)
        
        # The GPT Transformer Blocks
        self.blocks = nn.Sequential(*[GPTBlock(d_model, n_heads, dropout_prob) for _ in range(n_layers)])
        
        # Final Layer Norm and Language Modeling Head
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        
        # Weight tying (Improves performance and saves parameters)
        self.token_emb.weight = self.lm_head.weight

    def forward(self, idx, targets=None):
        B, T = idx.size()
        pos = torch.arange(0, T, dtype=torch.long, device=idx.device)
        
        # Token + Positional Embedding
        x = self.token_emb(idx) + self.pos_emb(pos)
        x = self.drop(x)
        
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        
        loss = None
        if targets is not None:
            # CrossEntropy expects (B*C, Classes)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
            
        return logits, loss

model = GPT10M()
print(f"Total Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.2f} M")

Total Parameters: 9.90 M


In [ ]:
class WikiDataset(Dataset):
    def __init__(self, file_path, tokenizer, seq_len=512):
        self.seq_len = seq_len
        self.data = []
        
        print("Tokenizing dataset line-by-line to save RAM...")
        with open(file_path, "r", encoding="utf-8") as f:
            # Process the file one line at a time
            for line in f:
                line = line.strip()
                if line:
                    # Encode the line and append ONLY the integer IDs
                    self.data.extend(tokenizer.encode(line).ids)
        
        # Convert the final list of IDs into a compact PyTorch tensor
        self.data = torch.tensor(self.data, dtype=torch.long)
        print(f"Total tokens in dataset: {len(self.data):,}")

    def __len__(self):
        return len(self.data) // self.seq_len

    def __getitem__(self, idx):
        # Slice out a chunk of seq_len + 1 (for the target)
        i = idx * self.seq_len
        chunk = self.data[i:i + self.seq_len + 1]
        x = chunk[:-1]
        y = chunk[1:]
        return x, y

seq_len = 256 
dataset = WikiDataset(txt_corpus_path, tokenizer, seq_len=seq_len)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True, num_workers=2)

Loading and tokenizing dataset into memory...


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=0.01)
scaler = torch.cuda.amp.GradScaler() # For Mixed Precision
epochs = 3

print("Starting Training...")
for epoch in range(epochs):
    model.train()
    total_loss = 0
    progress_bar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{epochs}")
    
    for batch_idx, (x, y) in enumerate(progress_bar):
        x, y = x.to(device), y.to(device)
        
        optimizer.zero_grad()
        
        # Mixed Precision Forward Pass
        with torch.cuda.amp.autocast():
            logits, loss = model(x, targets=y)
            
        # Backward Pass
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0) # Prevent exploding gradients
        scaler.step(optimizer)
        scaler.update()
        
        total_loss += loss.item()
        if batch_idx % 50 == 0:
            progress_bar.set_postfix({'loss': f"{loss.item():.4f}"})

    avg_loss = total_loss / len(dataloader)
    print(f"Epoch {epoch+1} Completed. Average Loss: {avg_loss:.4f}")

In [ ]:
import json

# 1. Save PyTorch Model State
model_path = "gpt_10m_model.pt"
torch.save(model.state_dict(), model_path)
print(f"Model saved to {model_path}")

# 2. Save a configuration file so you know the dimensions later
config = {
    "vocab_size": 32000,
    "d_model": 256,
    "n_heads": 8,
    "n_layers": 2,
    "max_seq_len": 512,
    "architecture": "GPT-Causal-LM"
}
with open("config.json", "w") as f:
    json.dump(config, f)

print("Training and Saving Complete. Files ready: tokenizer.json, gpt_10m_model.pt, config.json")